In [ ]:
# !pip install llama-index llama-index-vector-stores-chroma llama-index-embeddings-huggingface llama-index-llms-openai chromadb
# os.environ["OPENAI_API_KEY"] = "your-actual-openai-api-key-here"

import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
import chromadb

# 1. Initialize the LLM and the Embedding Model
# (Using a local Hugging Face embedding model and OpenAI for generation)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
llm = OpenAI(model="gpt-4o-mini")

# Configure global settings for LlamaIndex
from llama_index.core import Settings
Settings.llm = llm
Settings.embed_model = embed_model

# 2. Setup Vector Database (ChromaDB)
# This creates a persistent local database in the "./chroma_db" folder
db = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = db.get_or_create_collection("my_knowledge_base")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 3. Load Documents and Index Them
# Drops your PDFs/TXTs into a folder named "data"
documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

# 4. Create Query Engine and Search
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query("What are the key findings in the Q3 financial report?")

print("\n--- RAG Response ---")
print(response)